In [0]:
# =====================================================
# Imports
# =====================================================

import requests

import json

from requests.adapters import HTTPAdapter

from urllib3.util.retry import Retry

from uuid import uuid4

from dateutil.relativedelta import relativedelta

from pyspark.sql.functions import col

from delta.tables import DeltaTable

from datetime import (
    datetime,
    timezone,
    timedelta
)

from pyspark.sql.types import (
    StructType,
    StructField,
    DoubleType,
    StringType,
    TimestampType,
    LongType,
    IntegerType,
    DateType
)

In [0]:
# =====================================================
# Constants
# =====================================================

RETRY_STATUS_CODES = {
    "service_unavailable": 503,
    "gateway_timeout": 504,
    "too_many_requests": 429,
    "internal_service_error": 500,
    "bad_gateway": 502
}

API_URL = "https://archive-api.open-meteo.com/v1/archive"

CONFIG_TABLE = "weather_project.config.weather_locations"

HOURLY_VARIABLES = [
    "temperature_2m",
    "precipitation"
]
TEMPERATURE_UNIT = "celsius"
PRECIPITATION_UNIT = "mm"
TIME_FORMAT = "iso8601"

TIMEOUT_CONNECTION_SECONDS = 5
TIMEOUT_READ_SECONDS = 30

# Use ERA5 explicitly for a consistent reanalysis dataset across a longer historical window.
HISTORICAL_MODEL = "era5"

MODEL_PUBLICATION_LAG_DAYS = 5

PIPELINE_NAME = "openmeteo_era5_hourly"

PIPELINE_STATE_TABLE = "weather_project.control.pipeline_state"

BRONZE_TABLE_NAME = "weather_project.north_texas_weather.bronze_weather_raw"

In [0]:
# =====================================================
# Configure Spark
# =====================================================

spark.conf.set("spark.sql.session.timeZone", "UTC")

In [0]:
# ==========================================================================================================
# Call open-meteo API to retrieve weather data for each location in the configuration table.
# ==========================================================================================================

# =====================================================
# 1. Retry behavior
# =====================================================

retry_strategy = Retry(
    total=5,
    connect=5,
    read=5,
    status=5,
    backoff_factor=1,
    status_forcelist=list(RETRY_STATUS_CODES.values()),
    allowed_methods=["GET"],
    respect_retry_after_header=True,
    raise_on_status=False
)

session = requests.Session()

adapter = HTTPAdapter(max_retries=retry_strategy)

session.mount("https://", adapter)
session.mount("http://", adapter)

# =====================================================
# 2. Request open-meteo API
# =====================================================

location_dataframe = (
    spark.read
    .table(CONFIG_TABLE)
    .filter(col("active") == True)
    .orderBy("request_order")
)

locations = location_dataframe.collect()

if not locations:
    raise RuntimeError("No active locations found in weather configuration.")

state_row = (
    spark.read
    .table(PIPELINE_STATE_TABLE)
    .filter(
        col("pipeline_name") == PIPELINE_NAME
    )
    .first()
)

# Exclude the most recent five days because ERA5 is published with a documented five-day availability lag.
latest_available_date = datetime.now(timezone.utc).date() - timedelta(days=MODEL_PUBLICATION_LAG_DAYS)

if state_row is None:

    request_start_date = latest_available_date - relativedelta(years=5)

    print(f"No existing pipeline watermark found. Running initial five-year backfill.")

else:

    last_successful_data_date = state_row["last_successful_data_date"]

    request_start_date = last_successful_data_date + timedelta(days=1)

    print(f"Incremental ingestion detected. Last successful source date: {last_successful_data_date}")

request_end_date = latest_available_date

if request_start_date > request_end_date:

    print(f"No new ERA5 data is currently available. Pipeline is up to date through {last_successful_data_date}")

    # Pass run state to downstream Databricks Job tasks.
    dbutils.jobs.taskValues.set(
        key="has_new_data",
        value=False
    )

    dbutils.jobs.taskValues.set(
        key="ingestion_id",
        value=""
    )

    dbutils.notebook.exit("No new data available.")

print(f"Production pipeline initiated. Rolling window: {request_start_date} to {request_end_date}")

latitudes = [
    location["latitude"]
    for location in locations
]

longitudes = [
    location["longitude"]
    for location in locations
]

params = {
    "latitude": latitudes,
    "longitude": longitudes,
    "start_date": request_start_date.isoformat(),
    "end_date": request_end_date.isoformat(),
    "timezone": "UTC", # Open-meteo may label responses with GMT, but they have a zero UTC offset
    "models": HISTORICAL_MODEL,
    "temperature_unit": TEMPERATURE_UNIT,
    "precipitation_unit": PRECIPITATION_UNIT,
    "timeformat": TIME_FORMAT,
    "hourly": HOURLY_VARIABLES
}

response = session.get(API_URL, params=params, timeout=(TIMEOUT_CONNECTION_SECONDS, TIMEOUT_READ_SECONDS))

response.raise_for_status()

raw_data = response.json()
raw_response = response.text

display(raw_data)

In [0]:
# =====================================================
# 1. Ingestion metadata
# =====================================================

ingestion_id = str(uuid4())
ingested_at = datetime.now(timezone.utc)
http_status_code = response.status_code
response_content_type = response.headers.get("Content-Type")
response_size_bytes = len(response.content)
request_parameters = json.dumps(params, sort_keys=True)

# Snapshot location configuration in case someone changes the order of the locations or adds more cities
request_locations = [
    {
        "location_id": location["location_id"],
        "city": location["city"],
        "state_code": location["state_code"],
        "country_code": location["country_code"],
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "request_order": location["request_order"]
    }
    for location in locations
]

# =====================================================
# 2. Build ONE Bronze record per API request
# =====================================================

bronze_records = [
    {
        "ingestion_id": ingestion_id,
        "ingested_at": ingested_at,
        "source": "open-meteo",
        "endpoint": API_URL,
        "model": HISTORICAL_MODEL,
        "request_start_date": request_start_date,
        "request_end_date": request_end_date,
        "request_parameters": request_parameters,
        "http_status_code": http_status_code,
        "response_content_type": response_content_type,
        "response_size_bytes": response_size_bytes,
        "request_locations": json.dumps(request_locations, sort_keys=True),
        "raw_response": raw_response, # Store whole response string for stronger source fidelity
    }
]

# An explicit schema is created for the metadata
# The raw response is stored as a string for stronger source fidelity
# The bronze table stores ONE row per HTTP ingestion event

bronze_schema = StructType([
    StructField("ingestion_id", StringType(), False),
    StructField("ingested_at", TimestampType(), False),
    StructField("source", StringType(), False),
    StructField("endpoint", StringType(), False),
    StructField("model", StringType(), False),
    StructField("request_start_date", DateType(), False),
    StructField("request_end_date", DateType(), False),
    StructField("request_parameters", StringType(), False),
    StructField("http_status_code", IntegerType(), False),
    StructField("response_content_type", StringType(), False),
    StructField("response_size_bytes", LongType(), False),
    StructField("request_locations", StringType(), False),
    StructField("raw_response", StringType(), False)
])

df_bronze = spark.createDataFrame(bronze_records, schema=bronze_schema)

display(df_bronze)

(
    df_bronze
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(BRONZE_TABLE_NAME)
)

In [0]:
state_update_schema = StructType([
    StructField("pipeline_name", StringType(), False),
    StructField("source", StringType(), False),
    StructField("model", StringType(), False),
    StructField("last_successful_data_date", DateType(), False),
    StructField("last_successful_ingestion_id", StringType(), False),
    StructField("updated_at", TimestampType(), False),
])

state_update = spark.createDataFrame(
    [
        (
            PIPELINE_NAME,
            "open-meteo",
            HISTORICAL_MODEL,
            request_end_date,
            ingestion_id,
            datetime.now(timezone.utc)
        )
    ],
    schema=state_update_schema
)

state_target = DeltaTable.forName(spark, PIPELINE_STATE_TABLE)

(
    state_target
    .alias("target")
    .merge(
        state_update.alias("source"),
        """
        target.pipeline_name = source.pipeline_name
        AND target.source = source.source
        AND target.model = source.model
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
# =====================================================
# Pass Bronze ingestion metadata to downstream tasks
# =====================================================

dbutils.jobs.taskValues.set(
    key="has_new_data",
    value=True
)

dbutils.jobs.taskValues.set(
    key="ingestion_id",
    value=ingestion_id
)

print(
    f"Bronze ingestion completed successfully. "
    f"ingestion_id={ingestion_id}"
)